# Step 10 — **SAM3D from gsplat-rendered views** (AffordSplat / 3DGS → mesh + latents)

**Pipeline position:** after **`03_rendering_gaussian_splat.ipynb`** (or any path to a 3DGS `.ply`), before VLM/projection if you want mesh-centric features.

**What this does:**
1. **True gsplat** rasterisation → `run_dir/sam3d_dataset/images/view_XXX.png` + full-foreground masks (SAM3D layout). Cameras use the same **world-orbit + +90° ring (about +X)** and **`(-45°, 0°, +45°)`** ring azimuths as notebooks **09 / 11** (`ViewSelectionConfig` defaults).
2. **SAM3D** on **`view_002.png`** by default — that is the **+45°** ring-azimuth view (single-image API; `view_000` / `view_001` are −45° / 0° for debugging).
3. Writes **`run_dir/reconstruction/`** (`mesh.glb`, `gaussian.ply`, `shape_latent.pt`, …) and optionally **`data/cache/sam3d/<stem>/global_latent.pt`** per `configs/default.yaml`. On export, **`gsplat_ply_to_sam3d_reconstruction`** applies the **inverse** of `orbit_ring_rotation_deg` about `orbit_ring_rotation_axis` to the decoded mesh (and SAM3D Gaussian when `pytorch3d` is available) so disk artifacts align with the **normalized splat / GT** frame, not the tilted camera ring.
4. **Preview** (last cells): reference RGB; mesh as **pyrender** RGB composited on a light background, **`plt.imshow`** (works in Cursor), **`reconstruction/preview_sam3d_mesh.png`** on disk, and optional `IPython.display.Image`.

SAM3D prepends its checkout on `sys.path` only for the SAM3D call; `gsplat_ply_to_sam3d_reconstruction` removes those entries afterward so imports resolve to **this repo’s `src/`** again. **`LIDRA_SKIP_INIT`** is restored after SAM3D (only needed while the submodule loads). **`CUDA_HOME`** may remain defaulted from your conda env if it was unset before.

If **gsplat previews look gray** but another viewer shows color, your `.ply` may use a different `f_rest_*` column order — try `AFFORDANCE_GSPLAT_F_REST_LAYOUT=rgb_interleaved` before loading (see `rendering/gaussian_ply.py`).

**Requirements:** CUDA + **`gsplat`** + **SAM3D weights** (`sam-3d-objects/…`, `reconstruction.sam3d_config`). The last cell catches failures and prints the error.

**You cannot use this outside the container** in a supported way: the gsplat → SAM3D stack is pinned and tested only in the **`sam3d-pipeline` Docker** image (see [docker/README.md](../docker/README.md)). Run this notebook or the CLI **inside** `docker compose run --rm sam3d-pipeline bash` with the repo mounted at `/workspace`.

**CLI (from `/workspace` in the container):** `PYTHONPATH=src python scripts/render_gsplat_and_sam3d.py --splat_path … --run_dir …`

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for ROOT in [_cwd, *_cwd.parents]:
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
        if str(ROOT / "src") not in sys.path:
            sys.path.insert(0, str(ROOT / "src"))
        break
else:
    raise RuntimeError("Run from the repo or notebooks/.")

from utils.config import load_config

_cfg = load_config()
print("ROOT:", ROOT)


In [ ]:
# --- Edit: input splat and run directory (under exports/ so Cursor shows outputs) ---
from dataclasses import replace
from pathlib import Path

import torch

from datasets.affordsplat_local_dataset import resolve_affordsplat_root, sample_random_affordsplat_row
from rendering.renderer import build_render_config

RUN_STEM = "demo_gsplat_sam3d"
RUN_DIR = ROOT / "exports" / "gsplat_sam3d_runs" / RUN_STEM

# Same convention as notebooks 09 / 11: world-up azimuth ring, +90° about +X to align splat up,
# then fixed ring azimuths (-45°, 0°, +45°). Index 2 is the +45° view (SAM3D reference).
REFERENCE_VIEW = 2
GSPLAT_SAM3D_MRC = replace(
    build_render_config(_cfg),
    num_views=3,
    orbit_axis_mode="world",
    orbit_axis=None,
    orbit_ring_rotation_deg=90.0,
    orbit_ring_rotation_axis=(1.0, 0.0, 0.0),
    orbit_azimuth_offsets_deg=(-45.0, 0.0, 45.0),
)

# Set explicitly, or leave None to sample a random AffordSplat training Gaussian when mirror exists.
SPLAT_PLY: Path | None = None

_spl = SPLAT_PLY
if _spl is None:
    root = resolve_affordsplat_root(_cfg)
    if root is not None:
        row = sample_random_affordsplat_row(cfg=_cfg, subset="Seen", split="train", seed=None)
        if row is not None:
            _spl = row.splat_path
            print("Auto splat:", _spl, "|", row.sample_id)
if _spl is None or not Path(_spl).is_file():
    raise FileNotFoundError(
        "Set SPLAT_PLY to a .ply, or install AffordSplat under AFFORDANCE_DATA_ROOT / /workspace/data."
    )

SPLAT_PLY = Path(_spl).resolve()
print("CUDA:", torch.cuda.is_available(), "| SPLAT_PLY:", SPLAT_PLY)
print("RUN_DIR:", RUN_DIR.resolve())


In [ ]:
from reconstruction.gsplat_to_sam3d import gsplat_ply_to_sam3d_reconstruction

RUN_DIR.mkdir(parents=True, exist_ok=True)

try:
    out = gsplat_ply_to_sam3d_reconstruction(
        SPLAT_PLY,
        RUN_DIR,
        cfg=_cfg,
        object_stem=None,
        mrc=GSPLAT_SAM3D_MRC,
        reference_view_index=REFERENCE_VIEW,
        gsplat_seed=None,
        sam3d_seed=42,
        cache_global_latent=None,
    )
except Exception as exc:
    print("gsplat / SAM3D failed:", type(exc).__name__, exc)
    hint = (
        "Check: CUDA + gsplat for raster; SAM3D checkout with notebook/inference.py "
        "(git submodule update --init sam-3d-objects or SAM3D_OBJECTS_ROOT); "
        "HF checkpoints under reconstruction.sam3d_config (sam-3d-objects/doc/setup.md §2)."
    )
    if isinstance(exc, ModuleNotFoundError) and getattr(exc, "name", None) == "inference":
        hint = (
            "SAM3D code path: clone/init sam-3d-objects (notebook/inference.py) or set SAM3D_OBJECTS_ROOT; "
            "Docker bind mount must not hide /workspace/sam-3d-objects with an empty folder. "
            "Then HF checkpoints (reconstruction.sam3d_config)."
        )
    elif isinstance(exc, FileNotFoundError) and "SAM3D pipeline config" in str(exc):
        hint = ""  # Exception text already has HF download commands
    elif isinstance(exc, FileNotFoundError) and "pipeline.yaml" in str(exc):
        hint = (
            "Missing SAM3D weights: download facebook/sam-3d-objects on HF into "
            "sam-3d-objects/checkpoints/hf/ (see sam-3d-objects/doc/setup.md §2; hf auth login / HF_TOKEN)."
        )
    if hint:
        print(hint)
else:
    print("dataset_dir:", out["dataset_dir"])
    print("reconstruction_dir:", out["reconstruction_dir"])
    print("mesh:", out["paths"]["mesh"])
    if "global_latent_path" in out:
        print("global_latent:", out["global_latent_path"])
    print()
    print("Downstream: in notebooks 02, 04, 05, 06 set")
    print("  SAM3D_RUN_DIR =", repr(str(RUN_DIR.resolve())))
    print("then clear outputs/notebooks/02_rendering/ if you switch meshes, and rerun 02→04→05→06.")


In [ ]:
# Optional: show the reference RGB written for SAM3D
from IPython.display import Image, display

ref = RUN_DIR / "sam3d_dataset" / "images" / f"view_{REFERENCE_VIEW:03d}.png"
if ref.is_file():
    display(Image(filename=str(ref)))
else:
    print("No preview (run previous cell successfully first).")


In [ ]:
# SAM3D reconstruction mesh (RUN_DIR/reconstruction/mesh.glb)
# Show preview via matplotlib 2D (imshow) + write a PNG — Cursor often does not render
# IPython.display.Image(); the PNG path is always openable in the host image viewer.
import matplotlib.pyplot as plt
import numpy as np
import trimesh
from IPython.display import Image, display
from PIL import Image as PILImage

from datasets.mesh_loading import mesh_data_from_trimesh
from reconstruction.mesh_utils import reconstruction_paths
from rendering.mesh_renderer import MeshRenderConfig, MeshRenderer


def _as_trimesh(loaded):
    if isinstance(loaded, trimesh.Scene):
        parts = []
        for g in loaded.geometry.values():
            if isinstance(g, trimesh.Trimesh) and len(g.vertices) and len(g.faces):
                parts.append(g)
        if not parts:
            raise ValueError("empty glb Scene (no triangle geometry)")
        return trimesh.util.concatenate(parts) if len(parts) > 1 else parts[0]
    if isinstance(loaded, trimesh.Trimesh) and len(loaded.faces):
        return loaded
    raise TypeError(f"unexpected mesh type: {type(loaded).__name__}")


recon_dir = RUN_DIR / "reconstruction"
mesh_glb = reconstruction_paths(recon_dir)["mesh"]
preview_png = recon_dir / "preview_sam3d_mesh.png"

if not mesh_glb.is_file():
    print("No mesh yet — run the SAM3D cell successfully first:", mesh_glb)
else:
    mesh_vis = _as_trimesh(trimesh.load(str(mesh_glb), process=False))
    n_v, n_f = len(mesh_vis.vertices), len(mesh_vis.faces)
    print(f"mesh.glb: {n_v} vertices, {n_f} faces.")

    mesh_data = mesh_data_from_trimesh(mesh_vis)
    preview_mrc = MeshRenderConfig(
        image_size=640,
        fov_deg=55.0,
        num_views=1,
        camera_radius=2.2,
        elevation_deg=38.0,
        elevation_min_deg=25.0,
        elevation_max_deg=60.0,
        orbit_axis_mode="world",
        orbit_axis=None,
        orbit_ring_rotation_deg=0.0,
        render_geometry_aux=False,
    )
    try:
        views = MeshRenderer(preview_mrc).render(mesh_data)
        rgb = np.asarray(views[0].rgb)
        if rgb.ndim != 3 or rgb.shape[2] < 3:
            raise RuntimeError(f"unexpected RGB shape {getattr(rgb, 'shape', None)}")
        rgb = rgb[:, :, :3].astype(np.uint8, copy=False)
        depth = np.asarray(views[0].depth, dtype=np.float32)
        fg = depth > 0
        if float(rgb.mean()) < 15.0 and fg.any():
            bg = np.uint8([238, 238, 242])
            rgb = np.where(fg[..., None], rgb, bg)
    except Exception as exc:
        print("pyrender preview failed (EGL / GPU / headless):", type(exc).__name__, exc)
        print("Open mesh.glb externally, or run inside Docker sam3d-pipeline with working EGL/OSMesa.")
    else:
        PILImage.fromarray(rgb).save(preview_png)
        print("Wrote:", preview_png.resolve(), "| mean pixel:", float(rgb.mean()))
        if float(rgb.mean()) < 8.0:
            print(
                "WARN: preview still very dark — try opening the PNG above in your OS viewer, "
                "or set PYOPENGL_PLATFORM=osmesa in a headless environment."
            )

        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        ax.imshow(rgb)
        ax.set_axis_off()
        ax.set_title("SAM3D mesh (pyrender → PNG)")
        plt.tight_layout()
        plt.show()

        try:
            display(Image(filename=str(preview_png)))
        except Exception:
            pass